In [1]:
from osgeo import gdal
import os

In [2]:
"""Take the folders in case there is more than one folder"""

def list_folders(main_path):
    """
    Return a list of absolute paths (with forward slashes)
    of all immediate subfolders inside main_path.
    """
    folder_list = []

    for entry in os.listdir(main_path):
        full_path = os.path.join(main_path, entry)
        if os.path.isdir(full_path):
            folder_list.append(full_path.replace("\\", "/"))

    # If main_path is a folder, add it to the list
    if not folder_list:
        folder_list = [main_path]

    return folder_list


def find_folders_tifs(main_path, suffix = None): 
    """ 
    Recursively search inside main_path and all its subfolders and return a list of all .tif files found. 
    """ 
    folder_tif_dict = {}
    
    tif_files = [] 
    
    for root, dirs, files in os.walk(main_path):
        folder_name = os.path.basename(root)
        
        # Apply suffix filter only if provided
        if suffix is None or folder_name.endswith(suffix):
            for f in files: 
                if f.lower().endswith(".tif") or f.lower().endswith(".tiff"): 
                    full_path = os.path.join(root, f) 
                    tif_files.append(full_path.replace("\\", "/"))
                    # Create a dictionary with the folder name as key and the list of tif files as value
                    folder_tif_dict[folder_name] = tif_files
                      
    return tif_files, folder_tif_dict

def quality_check(File_list):
    
    corrupted_files = {}
    nodata_files = {}
    
    # Get the raster band
    for file in File_list[:]:
        # Open the raster file
        dataset = gdal.Open(file)
        if dataset is None:
            corrupted_files[file] = "Unable to open raster file"
        else:
            # Get the raster band
            band = dataset.GetRasterBand(1)  # Assuming you are working with the first band
            if band is None:
                corrupted_files[file] = "Unable to access raster band"
            else:
                # Get the nodata value of the band
                nodata_value = band.GetNoDataValue()
                if nodata_value is None:
                    nodata_files[file] = "The nodata value is not set"
                    
    return corrupted_files, nodata_files

In [3]:
"""Inputs"""

main_path = r"Z:\veg_c_storage_rawdata\testing\merge"

In [ ]:
"""Get all the data"""

# We get one folder or a list of the subfolders
Folder_list = list_folders(main_path)

# Check if it takes all of them
File_list, folder_tif_dict = find_folders_tifs(main_path, suffix = None)
print(len(File_list))
print(folder_tif_dict)

In [ ]:
"""This is a quality check for the nodata value"""

corrupted_files, nodata_files = quality_check(File_list)
print(corrupted_files, nodata_files)

The nodata value is not set.


In [11]:
"""
- Get a list of the raster files inside the folder.
- Do the geoprocessing per each file
"""
#beware if the folder indexation is okay
for folder in Folder_list[:]: #[:10]
    print("starting: " + folder)
    File_list = [] #f for f in os.listdir(path) if os.isfile(mypath,f)
    for root, dirs, files in os.walk(main_path):
        for f in files:
            if f.lower().endswith(".tif") or f.lower().endswith(".tiff"):
                full_path = os.path.join(root, f)
                File_list.append(full_path.replace("\\", "/"))
        else:
            pass
        
    output = os.path.basename(folder) + "_merged_total.tif" #if the output is the name of the folder
    # for a single output
    # output = "vcs_2020_global_300m_1.tif"

    # Specify the desired pixel size in the output
    # x_resolution = 100  # Horizontal pixel size
    # y_resolution = 100  # Vertical pixel size

    """Geoprocessing"""
    print("Merging raster files...")
    merged_tif = gdal.Warp(output, File_list, format="GTiff",
            # outputType = gdal.GDT_Byte, # this thing converts nodata values to zeros
            # srcNodata = -32768, # gives a value to the nodata areas
            # srcWin = [str(180), str(84), str(-180), str(-57)]
            # xRes=x_resolution, yRes=y_resolution,
            dstNodata = -9999, # sets the no data value
            # dstSRS = 'EPSG:3035',
            creationOptions=["COMPRESS=DEFLATE", "TILED=YES"])
    
    """remove the color pallete""" # optional         
    # band = merged_tif.GetRasterBand(1)
    # band.SetRasterColorTable(None)
    # # Close file and flush to disk
    # del band

    merged_tif = None 
    print(output)

    #time = 396

starting: Z:\z_resources\un_gbf\coastal protection\supply
Merging raster files...
supply_merged_total.tif


In [ ]:
"""File parser / For testing"""

path = r"Z:\z_resources\ruben\utci_2021\raster_output"
File_list = [] #f for f in os.listdir(path) if os.isfile(mypath,f)
for file in os.listdir(path): 
    if ".tif" in file:
        if file not in File_list:
            # File_list.append(os.path.join(path, file).replace("\\","/"))
            File_list.append(file)
    else:
        pass
    
    file_string = " ".join(File_list)
    print(file_string)

In [17]:
os.chdir(path)
text_file = open("sample.txt", "w")
n = text_file.write(file_string)
text_file.close()

In [6]:
print(os.getcwd())

\\akif.internal\public\z_resources\ruben\wb_temporal\cstorage_gabon2020
